In [2]:
import pandas as pd
import duckdb


In [3]:
orders = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "customer": ["Alex", "Maria", "James", "Alex", "Maria", "James", "Alex", "Maria"],
    "order_date": pd.to_datetime([
        "2026-01-05",
        "2026-01-07",
        "2026-01-10",
        "2026-02-02",
        "2026-02-15",
        "2026-03-01",
        "2026-03-20",
        "2026-04-05"
    ]),
    "total_sales": [600, 1200, 300, 900, 500, 700, 400, 1000]
})

display(orders)

,order_id,customer,order_date,total_sales
0,1001,Alex,2026-01-05,600
1,1002,Maria,2026-01-07,1200
2,1003,James,2026-01-10,300
3,1004,Alex,2026-02-02,900
4,1005,Maria,2026-02-15,500
5,1006,James,2026-03-01,700
6,1007,Alex,2026-03-20,400
7,1008,Maria,2026-04-05,1000


Hour 1 — SQL Date Functions

In [5]:
query = """
SELECT
    order_id,
    customer,
    order_date,
    EXTRACT(YEAR FROM order_date) AS order_year,
    EXTRACT(MONTH FROM order_date) AS order_month
FROM orders
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,order_date,order_year,order_month
0,1001,Alex,2026-01-05,2026,1
1,1002,Maria,2026-01-07,2026,1
2,1003,James,2026-01-10,2026,1
3,1004,Alex,2026-02-02,2026,2
4,1005,Maria,2026-02-15,2026,2
5,1006,James,2026-03-01,2026,3
6,1007,Alex,2026-03-20,2026,3
7,1008,Maria,2026-04-05,2026,4


Monthly Sales

How much revenue did we generate each month?

In [11]:
query = """
SELECT
    EXTRACT(YEAR FROM order_date) AS order_year,
    EXTRACT(MONTH FROM order_date) AS order_month,
    SUM(total_sales) AS monthly_sales
FROM orders
GROUP BY order_year, order_month
ORDER BY order_year, order_month
"""

result = duckdb.sql(query).df()
display(result)

,order_year,order_month,monthly_sales
0,2026,1,2100.0
1,2026,2,1400.0
2,2026,3,1100.0
3,2026,4,1000.0


Monthly Order Count

In [12]:
query = """
SELECT
    EXTRACT(YEAR FROM order_date) AS order_year,
    EXTRACT(MONTH FROM order_date) AS order_month,
    SUM(total_sales) AS monthly_sales,
    COUNT(order_id) AS order_count
FROM orders
GROUP BY order_year, order_month
ORDER BY order_year, order_month
"""

result = duckdb.sql(query).df()
display(result)

,order_year,order_month,monthly_sales,order_count
0,2026,1,2100.0,3
1,2026,2,1400.0,2
2,2026,3,1100.0,2
3,2026,4,1000.0,1


Average Order Value

In [16]:
query = """
SELECT
    EXTRACT(YEAR FROM order_date) AS order_year,
    EXTRACT(MONTH FROM order_date) AS order_month,
    SUM(total_sales) AS monthly_sales,
    COUNT(order_id) AS order_count,
    AVG(total_sales) AS avg_order_value
FROM orders
GROUP BY order_year, order_month
ORDER BY order_year, order_month
"""

result = duckdb.sql(query).df()
display(result)

,order_year,order_month,monthly_sales,order_count,avg_order_value
0,2026,1,2100.0,3,700.0
1,2026,2,1400.0,2,700.0
2,2026,3,1100.0,2,550.0
3,2026,4,1000.0,1,1000.0


Final Exercise: DATE_TRUNC()

In [18]:
query = """
SELECT
    DATE_TRUNC('month', order_date) AS order_month,
    SUM(total_sales) AS monthly_sales,
    COUNT(order_id) AS order_count,
    AVG(total_sales) AS avg_order_value
FROM orders
GROUP BY order_month
ORDER BY order_month
"""

result = duckdb.sql(query).df()
display(result)

,order_month,monthly_sales,order_count,avg_order_value
0,2026-01-01,2100.0,3,700.0
1,2026-02-01,1400.0,2,700.0
2,2026-03-01,1100.0,2,550.0
3,2026-04-01,1000.0,1,1000.0


Hour 2 — Comparing Events Across Time

How many days passed between each customer's orders?

In [ ]:
query = """
SELECT
    order_id,
    customer,
    order_date,
    LAG(order_date) OVER(
        PARTITION BY customer
        ORDER BY order_date)
    AS previous_order_date
FROM orders"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,order_date,previous_order_date
0,1003,James,2026-01-10,NaT
1,1006,James,2026-03-01,2026-01-10
2,1002,Maria,2026-01-07,NaT
3,1005,Maria,2026-02-15,2026-01-07
4,1008,Maria,2026-04-05,2026-02-15
5,1001,Alex,2026-01-05,NaT
6,1004,Alex,2026-02-02,2026-01-05
7,1007,Alex,2026-03-20,2026-02-02


Days Between Orders

How many days passed since each customer's previous order?

In [7]:
query = """
WITH order_history AS (
SELECT
    order_id,
    customer,
    order_date,
    LAG(order_date) OVER(
        PARTITION BY customer
        ORDER BY order_date)
    AS previous_order_date
FROM orders)

SELECT
    order_id,
    customer,
    order_date,
    previous_order_date,
    order_date - previous_order_date AS days_since_previous_order
FROM order_history
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,order_date,previous_order_date,days_since_previous_order
0,1002,Maria,2026-01-07,NaT,NaT
1,1005,Maria,2026-02-15,2026-01-07,39 days
2,1008,Maria,2026-04-05,2026-02-15,49 days
3,1001,Alex,2026-01-05,NaT,NaT
4,1004,Alex,2026-02-02,2026-01-05,28 days
5,1007,Alex,2026-03-20,2026-02-02,46 days
6,1003,James,2026-01-10,NaT,NaT
7,1006,James,2026-03-01,2026-01-10,50 days


Find the next order

In [9]:
query = """
SELECT
    order_id,
    customer,
    order_date,
    LEAD(order_date) OVER(
        PARTITION BY customer
        ORDER BY order_date)
    AS next_order_date
FROM orders"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,order_date,next_order_date
0,1002,Maria,2026-01-07,2026-02-15
1,1005,Maria,2026-02-15,2026-04-05
2,1008,Maria,2026-04-05,NaT
3,1001,Alex,2026-01-05,2026-02-02
4,1004,Alex,2026-02-02,2026-03-20
5,1007,Alex,2026-03-20,NaT
6,1003,James,2026-01-10,2026-03-01
7,1006,James,2026-03-01,NaT


Final Challenge

For each order, how many days until that customer's next order?

In [10]:
query = """
WITH future_order AS (
SELECT
    order_id,
    customer,
    order_date,
    LEAD(order_date) OVER(
        PARTITION BY customer
        ORDER BY order_date)
    AS next_order_date
FROM orders)

SELECT
    order_id,
    customer,
    order_date,
    next_order_date,
    next_order_date - order_date AS days_till_next_order
FROM future_order
"""

result = duckdb.sql(query).df()
display(result)

,order_id,customer,order_date,next_order_date,days_till_next_order
0,1003,James,2026-01-10,2026-03-01,50 days
1,1006,James,2026-03-01,NaT,NaT
2,1001,Alex,2026-01-05,2026-02-02,28 days
3,1004,Alex,2026-02-02,2026-03-20,46 days
4,1007,Alex,2026-03-20,NaT,NaT
5,1002,Maria,2026-01-07,2026-02-15,39 days
6,1005,Maria,2026-02-15,2026-04-05,49 days
7,1008,Maria,2026-04-05,NaT,NaT


Hour 3 — Pandas Datetime Work

In [11]:
orders["order_year"] = orders["order_date"].dt.year
orders["order_month"] = orders["order_date"].dt.month

Add these four columns:  
order_year  
order_month  
order_day  
order_day_name  

In [12]:
orders["order_day"] = orders["order_date"].dt.day

In [13]:
orders["order_day_name"] = orders["order_date"].dt.day_name()

In [14]:
orders.head()

,order_id,customer,order_date,total_sales,order_year,order_month,order_day,order_day_name
0,1001,Alex,2026-01-05,600,2026,1,5,Monday
1,1002,Maria,2026-01-07,1200,2026,1,7,Wednesday
2,1003,James,2026-01-10,300,2026,1,10,Saturday
3,1004,Alex,2026-02-02,900,2026,2,2,Monday
4,1005,Maria,2026-02-15,500,2026,2,15,Sunday


Monthly Sales in Pandas

In [15]:
orders.groupby(["order_year","order_month"]).agg(monthly_sales = ("total_sales","sum")).reset_index()

,order_year,order_month,monthly_sales
0,2026,1,2100
1,2026,2,1400
2,2026,3,1100
3,2026,4,1000


Multiple Monthly Metrics

In [16]:
monthly_summary = orders.groupby(
    ["order_year", "order_month"]
).agg(
    monthly_sales = ("total_sales", "sum"),
    order_count = ("order_id", "count"),
    avg_order_value = ("total_sales", "mean")
).reset_index()

In [17]:
display(monthly_summary)

,order_year,order_month,monthly_sales,order_count,avg_order_value
0,2026,1,2100,3,700.0
1,2026,2,1400,2,700.0
2,2026,3,1100,2,550.0
3,2026,4,1000,1,1000.0


Previous Order Date in Pandas

In [19]:
display(orders)

,order_id,customer,order_date,total_sales,order_year,order_month,order_day,order_day_name
0,1001,Alex,2026-01-05,600,2026,1,5,Monday
1,1002,Maria,2026-01-07,1200,2026,1,7,Wednesday
2,1003,James,2026-01-10,300,2026,1,10,Saturday
3,1004,Alex,2026-02-02,900,2026,2,2,Monday
4,1005,Maria,2026-02-15,500,2026,2,15,Sunday
5,1006,James,2026-03-01,700,2026,3,1,Sunday
6,1007,Alex,2026-03-20,400,2026,3,20,Friday
7,1008,Maria,2026-04-05,1000,2026,4,5,Sunday


In [20]:
orders = orders.sort_values(["customer","order_date"])

In [21]:
display(orders)

,order_id,customer,order_date,total_sales,order_year,order_month,order_day,order_day_name
0,1001,Alex,2026-01-05,600,2026,1,5,Monday
3,1004,Alex,2026-02-02,900,2026,2,2,Monday
6,1007,Alex,2026-03-20,400,2026,3,20,Friday
2,1003,James,2026-01-10,300,2026,1,10,Saturday
5,1006,James,2026-03-01,700,2026,3,1,Sunday
1,1002,Maria,2026-01-07,1200,2026,1,7,Wednesday
4,1005,Maria,2026-02-15,500,2026,2,15,Sunday
7,1008,Maria,2026-04-05,1000,2026,4,5,Sunday


In [22]:
orders["previous_order_date"] = (
    orders.groupby("customer")["order_date"].shift(1)
)

Final Exercise

In [26]:
orders["days_since_previous_order"] = (
    (orders["order_date"] - orders["previous_order_date"]).dt.days
)

In [27]:
display(orders)

,order_id,customer,order_date,total_sales,order_year,order_month,order_day,order_day_name,previous_order_date,days_since_previous_order
0,1001,Alex,2026-01-05,600,2026,1,5,Monday,NaT,NaN
3,1004,Alex,2026-02-02,900,2026,2,2,Monday,2026-01-05,28.0
6,1007,Alex,2026-03-20,400,2026,3,20,Friday,2026-02-02,46.0
2,1003,James,2026-01-10,300,2026,1,10,Saturday,NaT,NaN
5,1006,James,2026-03-01,700,2026,3,1,Sunday,2026-01-10,50.0
1,1002,Maria,2026-01-07,1200,2026,1,7,Wednesday,NaT,NaN
4,1005,Maria,2026-02-15,500,2026,2,15,Sunday,2026-01-07,39.0
7,1008,Maria,2026-04-05,1000,2026,4,5,Sunday,2026-02-15,49.0


Hour 4 — Project Data Quality

Load your Day 2 project data:

In [28]:
project_sales = pd.read_csv("../data/sales_analysis_clean.csv")

display(project_sales)
project_sales.info()

,order_id,customer_id,product_id,quantity,customer_name,state,product_name,price,total_sales
0,1001,1,10,2,Alex,CA,Laptop,1200,2400
1,1003,2,10,1,Maria,CA,Laptop,1200,1200
2,1005,3,11,2,James,TX,Monitor,300,600
3,1002,1,11,1,Alex,CA,Monitor,300,300
4,1004,3,12,3,James,TX,Keyboard,100,300


<class 'pandas.DataFrame'>
RangeIndex: 5 entries, 0 to 4
Data columns (total 9 columns):
 #   Column         Non-Null Count  Dtype
---  ------         --------------  -----
 0   order_id       5 non-null      int64
 1   customer_id    5 non-null      int64
 2   product_id     5 non-null      int64
 3   quantity       5 non-null      int64
 4   customer_name  5 non-null      str  
 5   state          5 non-null      str  
 6   product_name   5 non-null      str  
 7   price          5 non-null      int64
 8   total_sales    5 non-null      int64
dtypes: int64(6), str(3)
memory usage: 492.0 bytes


Check for duplicate order IDs

In [29]:
project_sales[
    project_sales.duplicated(subset=["order_id"], keep=False)
]

,order_id,customer_id,product_id,quantity,customer_name,state,product_name,price,total_sales


Missing Values

In [30]:
project_sales.isna().sum()

order_id         0
customer_id      0
product_id       0
quantity         0
customer_name    0
state            0
product_name     0
price            0
total_sales      0
dtype: int64

Invalid Values

In [31]:
project_sales[
    (project_sales["quantity"] <= 0) |
    (project_sales["price"] <= 0)
]

,order_id,customer_id,product_id,quantity,customer_name,state,product_name,price,total_sales


Validate total_sales

In [32]:
project_sales[
    project_sales["total_sales"] !=
    project_sales["quantity"] * project_sales["price"]
]

,order_id,customer_id,product_id,quantity,customer_name,state,product_name,price,total_sales


Final Challenge: Build a Data Quality Summary

In [33]:
duplicate_orders = project_sales.duplicated(subset=["order_id"]).sum()
missing_values = project_sales.isna().sum().sum()
invalid_quantity_or_price = (
    (project_sales["quantity"] <= 0) |
    (project_sales["price"] <= 0)
).sum()
incorrect_total_sales = (project_sales["total_sales"] !=
    project_sales["quantity"] * project_sales["price"]).sum()

Build the Summary

In [34]:
data_quality_summary = pd.DataFrame({
    "check": [
        "duplicate_orders",
        "missing_values",
        "invalid_quantity_or_price",
        "incorrect_total_sales"
    ],
    "issue_count": [
        duplicate_orders,
        missing_values,
        invalid_quantity_or_price,
        incorrect_total_sales
    ]
})

In [35]:
display(data_quality_summary)

,check,issue_count
0,duplicate_orders,0
1,missing_values,0
2,invalid_quantity_or_price,0
3,incorrect_total_sales,0


Save Your Work

In [36]:
data_quality_summary.to_csv(
    "../data/day_04_data_quality_summary.csv",
    index=False
)

Verify

In [37]:
check_day4 = pd.read_csv("../data/day_04_data_quality_summary.csv")

display(check_day4)

,check,issue_count
0,duplicate_orders,0
1,missing_values,0
2,invalid_quantity_or_price,0
3,incorrect_total_sales,0
